# 05 — Test Set Predictions

Generates submission.csv from the best trained SpectralTransformer checkpoint.
Runs inference on the full test set (10,000 samples) and saves predictions
for all gap positions (Is_Context=0).

Runtime: GPU (~few seconds)

In [1]:
# ── Cell 1: Setup ─────────────────────────────────────────────────────────

from google.colab import drive
drive.mount('/content/drive')

import os, sys

REPO = "https://github.com/Arjun11x/spectral-bridge.git"
if not os.path.exists('/content/spectral-bridge'):
    !git clone {REPO}
else:
    !cd /content/spectral-bridge && git pull

sys.path.insert(0, '/content/spectral-bridge')
!pip install -q tqdm

TRAIN_CSV = "/content/drive/MyDrive/spectral-bridge/data/spectral_graffiti.csv"
TEST_CSV  = "/content/drive/MyDrive/spectral-bridge/data/test_features_spectral.csv"

assert os.path.exists(TRAIN_CSV), f"Training CSV not found at {TRAIN_CSV}"
assert os.path.exists(TEST_CSV),  f"Test CSV not found at {TEST_CSV}"

print("=" * 50)
print("Setup complete!")
print(f"Train CSV : {TRAIN_CSV}")
print(f"Test CSV  : {TEST_CSV}")
print("=" * 50)

Mounted at /content/drive
Cloning into 'spectral-bridge'...
remote: Enumerating objects: 75, done.
remote: Counting objects: 100% (75/75), done.
remote: Compressing objects: 100% (58/58), done.
remote: Total 75 (delta 36), reused 54 (delta 15), pack-reused 0 (from 0)
Receiving objects: 100% (75/75), 1.59 MiB | 3.61 MiB/s, done.
Resolving deltas: 100% (36/36), done.
Setup complete!
Train CSV : /content/drive/MyDrive/spectral-bridge/data/spectral_graffiti.csv
Test CSV  : /content/drive/MyDrive/spectral-bridge/data/test_features_spectral.csv


In [2]:
# ── Cell 2: Imports and config ─────────────────────────────────────────────

import torch
from src import config
from src.utils import set_seed, get_device

set_seed(config.SEED)
device = get_device()

config.BEST_MODEL_PATH = "/content/drive/MyDrive/spectral-bridge/results/checkpoints/best_model.pth"

print(f"PyTorch version : {torch.__version__}")
print(f"Device          : {device}")
print(f"Model path      : {config.BEST_MODEL_PATH}")

Device: cuda — Tesla T4
PyTorch version : 2.10.0+cu128
Device          : cuda
Model path      : /content/drive/MyDrive/spectral-bridge/results/checkpoints/best_model.pth


In [3]:
# ── Cell 3: Generate test predictions ─────────────────────────────────────

from src.predict import predict

submission_path = predict(
    test_csv_path = TEST_CSV,
    output_path   = "/content/drive/MyDrive/spectral-bridge/results/submission.csv"
)

Device: cuda — Tesla T4
Loaded best model from /content/drive/MyDrive/spectral-bridge/results/checkpoints/best_model_best.pth
SpectralTransformer(d_model=64, n_heads=4, n_layers=3, d_ff=256, dropout=0.1, params=156,609)
Running inference on 10,000 test samples...
Inference complete.

Submission saved to   : /content/drive/MyDrive/spectral-bridge/results/submission.csv
Total gap predictions : 800,000
Unique samples        : 10,000
Value range           : -0.3335 to 1.3481

Sample preview:
 Sample_ID  Time_ms  Predicted_Value
     90000        1           0.4348
     90000        2           0.3844
     90000        3           0.3336
     90000        5           0.2548
     90000        6           0.2426
     90000        8           0.2647
     90000        9           0.2876
     90000       10           0.3147
     90000       11           0.3297
     90000       13           0.3163


In [4]:
# ── Cell 4: Preview and validate submission ────────────────────────────────

import pandas as pd

sub = pd.read_csv("/content/drive/MyDrive/spectral-bridge/results/submission.csv")

print(f"Submission shape      : {sub.shape}")
print(f"Total predictions     : {len(sub):,}")
print(f"Unique samples        : {sub['Sample_ID'].nunique():,}")
print(f"Predictions per sample: {len(sub) / sub['Sample_ID'].nunique():.0f}")
print(f"Value range           : {sub['Predicted_Value'].min():.4f} to {sub['Predicted_Value'].max():.4f}")
print(f"Missing values        : {sub.isnull().sum().sum()}")
print()
print("Sample preview:")
print(sub.head(10).to_string(index=False))

Submission shape      : (800000, 3)
Total predictions     : 800,000
Unique samples        : 10,000
Predictions per sample: 80
Value range           : -0.3335 to 1.3481
Missing values        : 0

Sample preview:
 Sample_ID  Time_ms  Predicted_Value
     90000        1           0.4348
     90000        2           0.3844
     90000        3           0.3336
     90000        5           0.2548
     90000        6           0.2426
     90000        8           0.2647
     90000        9           0.2876
     90000       10           0.3147
     90000       11           0.3297
     90000       13           0.3163
